In [ ]:
# Imports

import numpy as np
import sympy as sp
from sympy import symbols, Function, diff, tanh, sinh, exp, sqrt, simplify
from sympy.utilities.lambdify import lambdify
from scipy.integrate import solve_ivp
from pyswarms.single import GlobalBestPSO
import matplotlib.pyplot as plt


In [ ]:
# Symbolic variables and parameters

t = sp.symbols('t')
# Number of params to identify
n_vars = 8
kk = sp.symbols('k1:%d' % (n_vars+1))  # k1, k2, ..., k8

# Constants
Crate = -1

Numexp = 63
Totexp = 3100

N, M, NM = 2, 2, 2

# Design parameters
ep, es, en = 0.335, 0.47, 0.25
brugp, brugs, brugn = 2.43, 2.57, 2.91
lp, ls, ln1 = 75.6e-6, 12e-6, 85.2e-6
Rpp, Rpn = 5.22e-6, 5.86e-6
F = 96487
R_const = 8.3143
t1 = 0.363
ap = (3/Rpp)*(1-ep)
an = (3/Rpn)*(1-en)
T = 298.15
Acell = 0.11
Capa = 5
iapp = Capa * Crate / Acell

# Transport params symbolic with kk
c0 = 1000
D1 = kk[0] * 1e-9
Kappa = kk[1]
ctp = 51765
ctn = 29583
Dbulk = D1
sigmap = kk[2]
sigman = kk[3]
Dsp = kk[4] * 1e-15
Dsn = kk[5] * 1e-14

Keffp = Kappa * (ep ** brugp)
Keffs = Kappa * (es ** brugs)
Keffn = Kappa * (en ** brugn)
D2pos = (ep ** brugp) * Dbulk
D2sep = (es ** brugs) * Dbulk
D2neg = (en ** brugn) * Dbulk

kp = kk[6] * 1e-11
kn = kk[7] * 1e-12

h = lp/(N+1)
h2 = ls/(M+1)
h3 = ln1/(NM+1)

# Symbolic state variables X_i(t)
Nt = 1 + N + 1 + M + 1 + NM + 1 + N + NM + N + NM + N + 2 + NM + 2 + 1 + N + 1 + M + 1 + NM + 1

X = [Function(f'X_{i+1}')(t) for i in range(Nt)]

In [ ]:
# Create u1, u2, u3, u4, u5 arrays (mapping symbolic vars)

# Electrolyte concentration u1 (length = 1+N+1+M+1+NM+1)
u1_len = 1+N+1+M+1+NM+1
u1 = X[0:u1_len]

# Surface concentration u2 (length = N + NM)
u2 = [None] * (N + NM + 1)  # 1-based indexing adjustment, keep element 0 unused or None

for i in range(1, N+1):
    u2[i] = X[i + (N+1) + (M+1) + (NM+1) - 1]  # MATLAB 1-based to Python 0-based adjustment

for i in range(1, NM+1):
    u2[i+N] = X[i + (N+1) + (M+1) + (NM+1) + N - 1]

# Average concentration u3 (length = N + NM)
u3 = [None] * (N + NM + 1)
for i in range(1, N+1):
    u3[i] = X[i + (N+1) + (M+1) + (NM+1) + N + NM - 1]

for i in range(1, NM+1):
    u3[i+N] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + N - 1]

# Solid phase potential u4 (length = N + 2 + NM + 2)
u4 = [None] * (N + 2 + NM + 2)
for i in range(1, N+3):
    u4[i-1] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + N + NM - 1]

for i in range(1, NM+3):
    u4[i+N+1] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + N + NM + N + 2 - 1]

# Liquid potential u5 (length = 1 + N + 1 + M + 1 + NM + 1)
u5 = [None] * u1_len
offset = (N+1) + (M+1) + (NM+1) + N + NM + N + NM + N + 2 + NM + 2
for i in range(u1_len):
    u5[i] = X[i + offset]


In [ ]:
# Compute jp (molar flux) at positive electrode

jp = [None] * (N+2)  # 1-based, jp[0] unused

for i in range(1, N+2):
    theta = u2[i]  # theta = u2(i)*ctp/ctp = u2(i)
    Up = (-0.8090)*theta + 4.4875 - 0.0428*tanh(18.5138*(theta-0.5542)) - 17.7326*tanh(15.7890*(theta-0.3117)) + 17.5842*tanh(15.9308*(theta-0.3120))
    jp[i] = 2*kp*sqrt(u1[i]*c0)*sqrt(ctp - u2[i]*ctp)*sqrt(u2[i]*ctp)*sinh(0.5*F/(R_const*T)*(u4[i] - u5[i] - Up))


In [ ]:
#  Form PDE/ODE equations (example for electrolyte concentration in positive electrode)

# For example, finite difference spatial derivatives approximations
dudxf1 = ( -u1[2] - 3*u1[0] + 4*u1[1] ) / (2*h)  # Forward difference at boundary
dudxb1 = ( u1[N] + 3*u1[N+2] - 4*u1[N+1] ) / (2*h)  # Backward difference at boundary

# Initialize eq1 list
eq1 = [None] * u1_len

# Boundary condition at first node
eq1[0] = sp.Eq(0, dudxf1)

# Internal nodes (2 to N+1)
for i in range(1, N+1):
    d2udx21 = (u1[i-1] - 2*u1[i] + u1[i+1]) / h**2
    eq1[i] = sp.Eq(diff(u1[i], t), (D2pos*d2udx21 + ap*(1-t1)*jp[i]/c0)/ep)

# Boundary condition at node N+2
eq1[N+1] = sp.Eq(0, D2pos*dudxb1 - D2sep*dudxf1_2)


In [ ]:
# Numeric solution preparation (Mass matrix and ODE RHS functions)

# Given symbolic variables
t = sp.symbols('t')
kk = sp.symbols('kk0:8')  # kk0 to kk7, 8 parameters

# Assuming 'eqs' is a list of symbolic equations with lhs - rhs = 0 form
# And 'vars' is list of dependent variables: X_i(t)

# Rearrange eqs to get M * y_dot = f
# We separate time derivatives and algebraic parts
# Extract coefficients for derivatives and build M matrix and f vector

def mass_matrix_form(eqs, vars, t):
    """
    Given symbolic eqs of the form lhs == rhs,
    separate terms into M*y_dot = f
    Returns symbolic M matrix and f vector.
    """
    n = len(vars)
    M = sp.zeros(n)
    f_vec = sp.zeros(n, 1)

    for i, eq in enumerate(eqs):
        # lhs - rhs = 0  => eq = 0
        # Isolate derivatives (diff(vars[i], t)) terms
        # Collect coefficients
        eq = sp.simplify(eq)
        deriv = sp.Derivative(vars[i], t)
        coeff = eq.coeff(deriv)
        
        if coeff != 0:
            # Put coeff in M[i,i]
            M[i,i] = coeff
            # f_i = eq without derivative term
            f_vec[i] = eq - coeff * deriv
        else:
            # Algebraic eq (no derivative)
            M[i,i] = 0
            f_vec[i] = eq

    return M, f_vec

# Example usage
# M_sym, f_sym = mass_matrix_form(eqs, varsX, t)

# Now we convert M and f to numerical functions
# We must replace symbolic kk and vars with numerical arrays

from sympy.utilities.lambdify import lambdify

def generate_numeric_functions(M_sym, f_sym, varsX, kk):
    """
    Create callable numerical functions for M(t,y,kk) and f(t,y,kk)
    """
    # Flatten varsX and kk into list of symbols for lambdify
    variables = [t] + varsX + list(kk)
    
    # Lambdify M matrix and f vector element-wise (for efficiency)
    M_func = sp.lambdify(variables, M_sym, modules='numpy')
    f_func = sp.lambdify(variables, f_sym, modules='numpy')

    def M_numeric(time, y, params):
        args = [time] + list(y) + list(params)
        return np.array(M_func(*args), dtype=float)

    def f_numeric(time, y, params):
        args = [time] + list(y) + list(params)
        return np.array(f_func(*args), dtype=float).flatten()

    return M_numeric, f_numeric

# Then you would do:
# M_numeric, f_numeric = generate_numeric_functions(M_sym, f_sym, varsX, kk)


In [ ]:
U(1:1+N+1+M+1+NM+1) = 1;  % Electrolyte concentration
U(1+1+N+1+M+1+NM+1:N+1+N+1+M+1+NM+1) = 0.27;  % Surface concentration at positive
U(...) = ... % and so on


In [ ]:
# Parameter bounds for PSO (Particle Swarm Optimization)

# Define bounds arrays based on your MATLAB variables

pp = 0.3
D10 = 1
Kappa0 = 1.17
sigmap0 = 0.18
sigman0 = 215
Dsp0 = 4
Dsn0 = 3.3
kp0 = 0.7
kn0 = 0.7

lower_bound = np.array([D10*(1-pp), Kappa0*(1-pp), sigmap0*(1-pp), sigman0*(1-pp), Dsp0*(1-pp), Dsn0*(1-pp), kp0*(1-pp), kn0*(1-pp)])
upper_bound = np.array([D10*(1+pp), Kappa0*(1+pp), sigmap0*(1+pp), sigman0*(1+pp), Dsp0*(1+pp), Dsn0*(1+pp), kp0*(1+pp), kn0*(1+pp)])

bounds = (lower_bound, upper_bound)

# Define your objective function like P2Dobj in Python to run PSO
def objective_function(kk):
    # Run your ODE solver here with kk parameters, then calculate error (rms)
    # Return error
    pass

# Initialize optimizer
options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
optimizer = GlobalBestPSO(n_particles=10, dimensions=8, options=options, bounds=bounds)

best_cost, best_pos = optimizer.optimize(objective_function, iters=10)


In [ ]:
# ODE Solver Setup

def ode_system(t, y, params):
    M = M_numeric(t, y, params)
    f = f_numeric(t, y, params)
    # Solve M * y_dot = f => y_dot = M^{-1} * f
    y_dot = np.linalg.solve(M, f)
    return y_dot

# Define event function like stopcondition

def stop_condition(t, y, params):
    idx1 = 1+N+1+M+1+NM+1+N+NM+N+NM+1
    idx2 = 1+N+1+M+1+NM+1+N+NM+N+NM+N+2+NM+2
    return y[idx1] - y[idx2] - 2.7

stop_condition.terminal = True
stop_condition.direction = 0

# Solve ODE with solve_ivp

t_span = (0, 100000)  # Adjust as needed
y0 = ... # Your initial condition vector
params = best_pos  # Or initial guess

sol = solve_ivp(fun=lambda t,y: ode_system(t,y,params),
                t_span=t_span, y0=y0,
                method='BDF',
                events=lambda t,y: stop_condition(t,y,params),
                rtol=1e-5, atol=1e-5,
                max_step=5)

# sol.t, sol.y contain the solution


In [ ]:


plt.figure()
plt.plot(sol.t - 200, sol.y[idx1, :] - sol.y[idx2, :], linewidth=2, label='P2D Model')
time_exp = np.linspace(0, Totexp, Numexp)
plt.plot(time_exp, voltage_exp, 'o', markersize=7, color='red', label='Experiment')

plt.xlim([0, 3200])
plt.ylim([2.8, 4])
plt.xlabel('Time (seconds)', fontsize=15)
plt.ylabel('Voltage (V)', fontsize=15)
plt.legend()
plt.grid(True)
plt.show()
